# Candidate 2 Staged Validation Notebook: Soup Kernel Challenge

Kaggle settings (right panel):
- **Accelerator:** GPU T4 x2
- **Internet:** On (required for Hugging Face and package installation)

This notebook performs the staged validation of **Candidate 2** (`fix-rope-kernel-share`):
1. **Environment & Dependency Audit:** Verifies T4 GPU and pinned package versions.
2. **Repository Checkout:** Clones the candidate branch and optionally pins an immutable commit SHA.
3. **Gate 1 (Unit Tests):** Prepares task data and executes `pytest solution -v`.
4. **Gate 1B (RoPE GPU & PTX Diagnostic):** Runs RoPE on CUDA, tests failing FP16 case, realistic frequencies, FP32, backward gradients, and inspects PTX.
5. **Gate 2 (Qwen2MLP Equivalence):** Compares output, input gradient, and projection weight gradients against PyTorch reference.
6. **Gate 3 (Soup Regression Suite):** Runs official test suite with `SOUP_APPLY_SOLUTION=1`.
7. **Benchmark:** Executes `run_task.py --repeats 2` across baseline and solution.
8. **Scoring & Rule Evaluation:** Executes `check_rules.py` and `score.py` (scoring always runs even if Check 4 fails).

In [ ]:
# Candidate repository and branch configuration
# Note: The branch must be pushed to GitHub before Kaggle can clone it.
# For official evaluation, set COMMIT to the exact source commit SHA.
REPO_URL = "https://github.com/raulibaddin/soup-kernel-challenge-demo.git"
BRANCH = "fix-rope-kernel-share"
COMMIT = "9c19d49a4a49ac31b1176d7633d0d30ff8a38dda"  # Candidate 2 source commit SHA

In [ ]:
import os, platform, subprocess, sys

# 1. Hardware verification
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print("Detected GPU:", gpus.strip())
assert "T4" in gpus, "Accelerator must be GPU T4 x2"

# 2. Token setup
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

# 3. Pinned dependency installation matching competition specification
print("\nInstalling official pinned dependencies...")
install_cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "soup-cli[train]==0.77.0",
    "triton==3.6.0",
    "transformers==5.16.1",
    "peft==0.20.0",
    "trl==0.29.1",
    "bitsandbytes==0.50.2",
    "accelerate==1.14.0",
    "pytest"
]
subprocess.run(install_cmd, check=True)

# 4. Strict environment and package version verification
from importlib.metadata import version
import torch
import triton
import transformers
import peft
import trl
import bitsandbytes as bnb
import accelerate

soup_version = version("soup-cli")

print("\n=== Environment & Dependency Version Audit ===")
print("Python:       ", platform.python_version())
print("PyTorch:      ", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version: ", torch.version.cuda)
print("Device Name:  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
print("Device Count: ", torch.cuda.device_count())
print("Triton:       ", getattr(triton, "__version__", "unknown"))
print("Soup CLI:     ", soup_version)
print("Transformers: ", transformers.__version__)
print("PEFT:         ", peft.__version__)
print("TRL:          ", trl.__version__)
print("BitsAndBytes: ", bnb.__version__)
print("Accelerate:   ", accelerate.__version__)

# Verify official competition environment and pinned package requirements
assert (3, 11) <= sys.version_info[:2] < (3, 14), (
    f"Soup CLI 0.77.0 requires Python 3.11–3.13, "
    f"got {platform.python_version()}"
)
assert torch.cuda.is_available() and torch.version.cuda is not None, "CUDA-enabled PyTorch is required"
assert "T4" in torch.cuda.get_device_name(0), f"Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
assert soup_version == "0.77.0", f"Expected soup-cli==0.77.0, got {soup_version}"
assert getattr(triton, "__version__", "").split("+")[0] == "3.6.0", f"Expected triton==3.6.0, got {getattr(triton, '__version__', None)}"
assert transformers.__version__ == "5.16.1", f"Expected transformers==5.16.1, got {transformers.__version__}"
assert peft.__version__ == "0.20.0", f"Expected peft==0.20.0, got {peft.__version__}"
assert trl.__version__ == "0.29.1", f"Expected trl==0.29.1, got {trl.__version__}"
assert bnb.__version__ == "0.50.2", f"Expected bitsandbytes==0.50.2, got {bnb.__version__}"
assert accelerate.__version__ == "1.14.0", f"Expected accelerate==1.14.0, got {accelerate.__version__}"
# PyTorch compatibility floor: soup-cli[train] specification requires torch>=2.6.0 (no exact version is mandated by the competition brief)
torch_major_minor = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
assert torch_major_minor >= (2, 6), f"soup-cli train requires torch>=2.6.0 compatibility floor, got {torch.__version__}"
print("All official competition dependencies and pinned requirements verified successfully!")

In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("Cloning without GITHUB_TOKEN secret:", type(exc).__name__)

auth = []
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]

shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
clone_cmd = ["git", *auth, "clone", "-q", "-b", BRANCH, REPO_URL, "/kaggle/working/repo"]
clone = subprocess.run(clone_cmd, capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)

if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)

current_commit = subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                                capture_output=True, text=True).stdout
print("Active Repository Commit:", current_commit.strip())

In [ ]:
import subprocess
# Prepare task dataset
subprocess.run(["python", "task/prepare_data.py"], cwd="/kaggle/working/repo", check=True)

# Gate 1: Run custom kernel unit tests (capture exit code without immediately aborting)
print("Running custom kernel test suite (pytest solution -v)...")
proc_gate1 = subprocess.run(["python", "-m", "pytest", "solution", "-v"], cwd="/kaggle/working/repo")
gate1_rc = proc_gate1.returncode
print(f"\nGate 1 (pytest solution) exit code: {gate1_rc}")
if gate1_rc != 0:
    print("WARNING: Custom kernel unit tests failed; proceeding to Gate 1B RoPE GPU diagnostic before gating.")

In [ ]:
# Gate 1B: Focused RoPE GPU PTX and precision diagnostic on CUDA
rope_diag = '''
import torch, sys
sys.path.insert(0, '/kaggle/working/repo')
from solution.rope import apply_triton_rope, rope_fwd_kernel, rope_bwd_kernel
from solution.test_kernels import _reference_rope

print("=== RoPE GPU Diagnostic & PTX Inspection ===")

# 1. Inspect generated PTX if exposed by Triton compiled kernel
try:
    B, H, S, D = 4, 12, 512, 128
    HALF_D = D // 2
    x_test = torch.empty((B * H * S, D), device='cuda', dtype=torch.float16)
    cos_test = torch.empty((B * S, D), device='cuda', dtype=torch.float16)
    sin_test = torch.empty((B * S, D), device='cuda', dtype=torch.float16)
    out_test = torch.empty_like(x_test)
    grid = (B * H * S,)
    compiled = rope_fwd_kernel[grid](
        x_test, cos_test, sin_test, out_test,
        B * H * S, S, H * S,
        HAS_B_COS=True, D=D, HALF_D=HALF_D
    )
    ptx = compiled.asm.get('ptx', '')
    if ptx:
        fma_f16 = 'fma.rn.f16' in ptx or 'hfma2' in ptx
        fma_f32 = 'fma.rn.f32' in ptx
        mul_f16 = 'mul.rn.f16' in ptx or 'hmul2' in ptx
        add_f16 = 'add.rn.f16' in ptx or 'hadd2' in ptx
        print(f"PTX Inspection: fma.rn.f16/hfma2={fma_f16}, fma.rn.f32={fma_f32}, mul.f16/hmul2={mul_f16}, add.f16/hadd2={add_f16}")
    else:
        print("PTX inspection: asm dict did not contain 'ptx' key.")
except Exception as exc:
    print(f"PTX inspection note: {exc}")

tol_fp16 = 2e-3
tol_fp32 = 1e-4
failures = []

def audit_case(name, q, cos, sin, grad, tol):
    ref_q, _ = _reference_rope(q, q, cos, sin)
    ref_q.backward(grad)
    ref_dq = q.grad.clone()

    q_tri = q.detach().clone().requires_grad_(True)
    tri_q = apply_triton_rope(q_tri, cos, sin)
    tri_q.backward(grad)

    diff_fwd = (tri_q.float() - ref_q.float().detach()).abs()
    diff_bwd = (q_tri.grad.float() - ref_dq.float()).abs()
    
    thresh_fwd = tol + tol * ref_q.float().detach().abs()
    thresh_bwd = tol + tol * ref_dq.float().abs()

    mismatch_fwd = (diff_fwd > thresh_fwd).sum().item()
    mismatch_bwd = (diff_bwd > thresh_bwd).sum().item()
    total = diff_fwd.numel()

    max_abs_fwd = diff_fwd.max().item()
    max_rel_fwd = (diff_fwd / (ref_q.float().detach().abs() + 1e-5)).max().item()
    max_abs_bwd = diff_bwd.max().item()
    max_rel_bwd = (diff_bwd / (ref_dq.float().abs() + 1e-5)).max().item()

    status_fwd = 'PASS' if mismatch_fwd == 0 else f'FAIL ({mismatch_fwd}/{total} mismatches)'
    status_bwd = 'PASS' if mismatch_bwd == 0 else f'FAIL ({mismatch_bwd}/{total} mismatches)'
    print(f"{name}:")
    print(f"  Forward:  max abs = {max_abs_fwd:.8f}, max rel = {max_rel_fwd:.8f} [{status_fwd}]")
    print(f"  Backward: max abs = {max_abs_bwd:.8f}, max rel = {max_rel_bwd:.8f} [{status_bwd}]")
    if mismatch_fwd != 0 or mismatch_bwd != 0:
        failures.append(f"{name} (fwd={status_fwd}, bwd={status_bwd})")

# 2. Original failing test: FP16 random frequencies (B=4, H=12, S=512, D=128, seed 42)
torch.manual_seed(42)
B, H, S, D = 4, 12, 512, 128
q16 = torch.randn(B, H, S, D, device='cuda', dtype=torch.float16, requires_grad=True)
cos16 = torch.randn(B, S, D, device='cuda', dtype=torch.float16)
sin16 = torch.randn(B, S, D, device='cuda', dtype=torch.float16)
grad16 = torch.randn(B, H, S, D, device='cuda', dtype=torch.float16)
audit_case("1. FP16 Random Frequencies (Original Failing Shape)", q16, cos16, sin16, grad16, tol_fp16)

# 3. Realistic Qwen2 rotary frequencies (unit circle: cos^2 + sin^2 = 1)
inv_freq = 1.0 / (1000000.0 ** (torch.arange(0, D, 2, device='cuda', dtype=torch.float32) / D))
t = torch.arange(S, device='cuda', dtype=torch.float32)
freqs = torch.outer(t, inv_freq)
emb = torch.cat([freqs, freqs], dim=-1)
cos_real = emb.cos().unsqueeze(0).expand(B, -1, -1).half()
sin_real = emb.sin().unsqueeze(0).expand(B, -1, -1).half()
q16_r = torch.randn(B, H, S, D, device='cuda', dtype=torch.float16, requires_grad=True)
grad16_r = torch.randn(B, H, S, D, device='cuda', dtype=torch.float16)
audit_case("2. FP16 Realistic Rotary Frequencies", q16_r, cos_real, sin_real, grad16_r, tol_fp16)

# 4. FP32 Test
q32 = torch.randn(B, H, S, D, device='cuda', dtype=torch.float32, requires_grad=True)
cos32 = torch.randn(B, S, D, device='cuda', dtype=torch.float32)
sin32 = torch.randn(B, S, D, device='cuda', dtype=torch.float32)
grad32 = torch.randn(B, H, S, D, device='cuda', dtype=torch.float32)
audit_case("3. FP32 Precision Check", q32, cos32, sin32, grad32, tol_fp32)

# 5. Verify runtime dtypes from Transformers 5.16.1 Qwen2RotaryEmbedding on CUDA
try:
    from transformers.models.qwen2.configuration_qwen2 import Qwen2Config
    from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
    qcfg = Qwen2Config(hidden_size=1536, num_attention_heads=12, num_key_value_heads=2, max_position_embeddings=512, rope_parameters={'rope_type': 'default', 'rope_theta': 1000000.0})
    rot_emb = Qwen2RotaryEmbedding(qcfg).cuda()
    dummy_x = torch.randn(4, 512, 1536, device='cuda', dtype=torch.float16)
    dummy_pos = torch.arange(512, device='cuda').unsqueeze(0).expand(4, -1)
    cos_c, sin_c = rot_emb(dummy_x, dummy_pos)
    print(f"4. Qwen2RotaryEmbedding CUDA dtypes: cos={cos_c.dtype}, sin={sin_c.dtype}")
    if cos_c.dtype != torch.float16 or sin_c.dtype != torch.float16:
        failures.append(f'Runtime dtype mismatch: cos={cos_c.dtype}, sin={sin_c.dtype}')
except Exception as exc:
    failures.append(f'Runtime dtype check error: {exc}')

if failures:
    print()
    print(f'RoPE GPU diagnostic recorded {len(failures)} failure(s):')
    for f in failures:
        print(f'  - {f}')
    sys.exit(1)
else:
    print()
    print('All RoPE GPU equivalence and runtime dtype checks passed successfully!')
    sys.exit(0)
'''
proc_gate1b = subprocess.run([sys.executable, "-c", rope_diag])
gate1b_rc = proc_gate1b.returncode
print(f"\nGate 1B (RoPE GPU diagnostic) exit code: {gate1b_rc}")
if gate1b_rc != 0:
    print("WARNING: Gate 1B diagnostic recorded failures.")

In [ ]:
# Gate Check: Halt validation if preliminary kernel tests (Gate 1 or Gate 1B) failed
preliminary_failures = []
if 'gate1_rc' not in globals() or gate1_rc != 0:
    preliminary_failures.append(f"Gate 1 (pytest solution) failed with exit code {globals().get('gate1_rc', 'NOT_RUN')}")
if 'gate1b_rc' not in globals() or gate1b_rc != 0:
    preliminary_failures.append(f"Gate 1B (RoPE GPU diagnostic) failed with exit code {globals().get('gate1b_rc', 'NOT_RUN')}")
if preliminary_failures:
    raise RuntimeError("Validation HALTED before Gate 2: Preliminary kernel tests failed!\n" + "\n".join(f"  - {pf}" for pf in preliminary_failures) + "\nAborting MLP integration test, Soup regression suite, and benchmark to prevent running incorrect kernels.")
print("Preliminary kernel gates PASSED (Gate 1 exit code 0, Gate 1B exit code 0). Proceeding to Gate 2...")

# Gate 2: Comprehensive Qwen2MLP forward, backward, and parameter gradient equivalence on CUDA
mlp_diag = '''
import torch, sys
sys.path.insert(0, '/kaggle/working/repo')
from transformers.models.qwen2.configuration_qwen2 import Qwen2Config
from transformers.models.qwen2.modeling_qwen2 import Qwen2MLP
import solution

print("=== Qwen2MLP Forward, Backward & Parameter Gradient Audit ===")
print("NOTE: This test validates unquantized PyTorch Qwen2MLP forward, backward, and projection gradients.")
print("LoRA integration on 4-bit Linear4bit layers is validated end-to-end in Gate 3 and the full benchmark.")
print()

config = Qwen2Config(
    hidden_size=1536,
    intermediate_size=8960,
    hidden_act='silu',
)

torch.manual_seed(42)
mlp_ref = Qwen2MLP(config).cuda().half()
mlp_custom = Qwen2MLP(config).cuda().half()
mlp_custom.load_state_dict(mlp_ref.state_dict())

x_ref = torch.randn(4, 512, 1536, device='cuda', dtype=torch.float16, requires_grad=True)
x_custom = x_ref.detach().clone().requires_grad_(True)
grad = torch.randn(4, 512, 1536, device='cuda', dtype=torch.float16)

# 1. Execute unpatched reference forward & backward BEFORE applying solution
out_ref = mlp_ref(x_ref)
out_ref.backward(grad)

# 2. Apply solution patch to Qwen2MLP
solution.apply()

# 3. Verify custom kernel is executed and does not silently trigger fallback
called = {'swiglu': False}
orig_swiglu = solution.swiglu.swiglu
def tracked_swiglu(*args, **kwargs):
    called['swiglu'] = True
    return orig_swiglu(*args, **kwargs)
solution.swiglu.swiglu = tracked_swiglu

# 4. Execute patched forward & backward
out_custom = mlp_custom(x_custom)
out_custom.backward(grad)

assert called['swiglu'] is True, "Custom SwiGLU kernel was NOT called! CUDA guard triggered fallback!"

# 5. Detailed error comparison across outputs, input grads, and projection weight grads
comparisons = [
    ("MLP Forward Output", out_custom, out_ref),
    ("Input Grad (dx)", x_custom.grad, x_ref.grad),
    ("gate_proj.weight.grad", mlp_custom.gate_proj.weight.grad, mlp_ref.gate_proj.weight.grad),
    ("up_proj.weight.grad", mlp_custom.up_proj.weight.grad, mlp_ref.up_proj.weight.grad),
    ("down_proj.weight.grad", mlp_custom.down_proj.weight.grad, mlp_ref.down_proj.weight.grad),
]

tol = 2e-3
all_passed = True
print(f"{'Comparison':<25} | {'Max Abs Err':<12} | {'Max Rel Err':<12} | {'Mismatches (>tol)':<18} | {'Total Elements'}")
print("-" * 85)

for name, actual, expected in comparisons:
    diff = (actual.float() - expected.float()).abs()
    max_abs = diff.max().item()
    max_rel = (diff / (expected.float().abs() + 1e-5)).max().item()
    thresh = tol + tol * expected.float().abs()
    mismatches = (diff > thresh).sum().item()
    total = expected.numel()
    status = "OK" if mismatches == 0 else "FAIL"
    print(f"{name:<25} | {max_abs:<12.6f} | {max_rel:<12.6f} | {mismatches:<18} | {total:<12} [{status}]")
    if mismatches > 0:
        all_passed = False

assert all_passed, "Qwen2MLP integration test failed: element mismatches exceeded tolerance!"
print()
print("Qwen2MLP Real CUDA Equivalence Verified Successfully!")
'''
subprocess.run([sys.executable, "-c", mlp_diag], check=True)

In [ ]:
import os, subprocess
# Gate 3: Official Soup regression test suite (enforcing SOUP_APPLY_SOLUTION=1)
print("Running official Soup regression test suite with solution enabled...")
env = dict(os.environ, SOUP_APPLY_SOLUTION="1")
subprocess.run(["python", "-m", "pytest", "tests", "-q"], cwd="/kaggle/working/repo", env=env, check=True)

In [ ]:
import subprocess
# Step 4: Full official training benchmark (2 repeats: base, sol, sol, base)
print("Running full benchmark harness with 2 repeats...")
subprocess.run(["python", "harness/run_task.py", "--repeats", "2"], cwd="/kaggle/working/repo", check=True)

In [ ]:
import os, subprocess, sys

# Step 5: Official rule checks and score calculation
# Both check_rules.py and score.py run so results are always reported even if Check 4 fails.
print("=== Running check_rules.py ===")
proc_check = subprocess.run(["python", "harness/check_rules.py"], cwd="/kaggle/working/repo")
check_rc = proc_check.returncode

print("\n=== Running score.py ===")
result_json = "/kaggle/working/repo/runs/result.json"
result_exists = os.path.exists(result_json)
score_rc = None
if result_exists:
    proc_score = subprocess.run(["python", "harness/score.py", "runs/result.json"], cwd="/kaggle/working/repo")
    score_rc = proc_score.returncode
else:
    print("ERROR: runs/result.json does not exist; score.py cannot be executed.")

print("\n=== Final Execution Summary ===")
print(f"runs/result.json exists:  {result_exists}")
print(f"check_rules.py exit code: {check_rc}")
print(f"score.py exit code:       {score_rc}")

failures = []
if not result_exists:
    failures.append("runs/result.json does not exist (benchmark did not produce result file)")
if check_rc != 0:
    failures.append(f"check_rules.py failed with exit code {check_rc}")
if score_rc is None or score_rc != 0:
    failures.append(f"score.py failed with exit code {score_rc}")

if failures:
    raise RuntimeError("Validation FAILED with the following errors:\n" + "\n".join(f"  - {f}" for f in failures))

print("\nValidation completed successfully: all checks passed and scored!")